In [37]:
import numpy as np
import scipy.optimize as opt
from scipy import signal
from scipy.special import erfc, erfcinv

from optic.models.devices import mzm, photodiode
from optic.models.channels import linearFiberChannel
from optic.comm.modulation import modulateGray, grayMapping
from optic.comm.sources import bitSource, symbolSource
from optic.dsp.core import upsample, pulseShape, anorm, firFilter
from optic.utils import parameters, dBm2W
from optic.comm.metrics import bert


In [38]:
# simulation parameters
SpS = 16
Rs = 32e9
Fs = SpS * Rs
M = 16
nBits = 400000
rollOff = 0.01
nFilterTaps = 1024
mzmScale = 0.5
Vpi = 2
laserLinewidth = 100e3

paramSymb = parameters()
paramSymb.nSymbols = int(nBits // np.log2(M))
paramSymb.M = M
paramSymb.constType = "qam"
paramSymb.dist = "uniform"
paramSymb.seed = 444
paramSymb.shapingFactor = 0

# Pulse shaping
paramPulse = parameters()
paramPulse.pulseType = "rrc"
paramPulse.nFilterTaps = nFilterTaps
paramPulse.rollOff = rollOff
paramPulse.SpS = SpS

# IQM
paramIQM = parameters()
paramIQM.Vpi = Vpi
paramIQM.VbI = -Vpi
paramIQM.VbQ = -Vpi
paramIQM.Vphi = Vpi / 2

Pi_dBm = 3
Pi = dBm2W(Pi_dBm)

# MZM parameters
paramMZM = parameters()
paramMZM.Vpi = 2
paramMZM.Vb = -paramMZM.Vpi / 2

# fiber channel parameters
paramCh = parameters()
paramCh.L = 100
paramCh.alpha = 0.2
paramCh.D = 16
paramCh.Fc = 193.1e12
paramCh.Fs = Fs

# photodiode parameters
paramPD = parameters()
paramPD.ideal = False
paramPD.B = Rs
paramPD.Fs = Fs
paramPD.seed = 456

In [39]:
def rapp_pa(x, Vsat, p=2.0):
    mag = np.abs(x)
    gain = 1.0 / (1.0 + (mag / Vsat) ** (2 * p)) ** (1.0 / (2 * p))
    return x * gain

def butter_lpf_complex(x, Fs, f3dB, order=3):
    wn = f3dB / (Fs / 2)
    if wn >= 1.0:
        raise ValueError(
            f"Butterworth cutoff must be below Nyquist. Got f3dB={f3dB/1e9:.2f} GHz, "
            f"Fs/2={(Fs/2)/1e9:.2f} GHz."
        )
    b, a = signal.butter(order, wn, btype='low')
    y_i = signal.lfilter(b, a, np.real(x))
    y_q = signal.lfilter(b, a, np.imag(x))
    return y_i + 1j * y_q

def pa_model(x, Fs, BO_dB=5.0, p=2.0, f3dB=37e9):
    Vrms = np.sqrt(np.mean(np.abs(x) ** 2))
    Vsat = Vrms * 10 ** (BO_dB / 20)
    y_nl = rapp_pa(x, Vsat=Vsat, p=p)
    y = butter_lpf_complex(y_nl, Fs=Fs, f3dB=f3dB, order=3)
    return y

In [40]:
# PA parameters
PA_enable = True
PA_BO_dB = 5.0
PA_p = 2.0
PA_f3dB = 37e9

def optical_chain(symbTx):
    # upsampling
    symbolsUp = upsample(symbTx, SpS)

    # pulse shaping
    pulse = pulseShape(paramPulse)
    sigTx = firFilter(pulse, symbolsUp)
    sigTx = anorm(sigTx)  # normalize to 1 Vpp

    # PA distortion
    sigTx = pa_model(sigTx, Fs=Fs, BO_dB=5.0, p=2.0, f3dB=37e9)

    # optical modulation
    Ai = np.sqrt(Pi)
    sigTxo = mzm(Ai, sigTx, paramMZM)

    # linear fiber channel model
    sigCh = linearFiberChannel(sigTxo, paramCh)

    # noisy PD
    I_Rx = photodiode(sigCh, paramPD)

    # symbol-rate samples
    I_Rx = I_Rx[0::SpS]

    return I_Rx

In [41]:
#Utility Functions
# Helper functions for normalization and metrics
def crop_same_length(a, b):
    #Ensure two signals have the same length
    n = min(len(a), len(b))
    return np.asarray(a[:n]), np.asarray(b[:n])


def normalize_signal(x, eps=1e-12):
    #Normalize signal to zero mean and unit variance
    mu = np.mean(x)
    std = np.std(x) + eps
    return (x-mu)/std, mu, std


def apply_norm(x, mu, std):
    #Apply previously computed normalization
    return (x-mu)/(std+1e-12)


def nmse_db(y_true, y_pred):
    #Compute normalized MSE in dB
    y_true, y_pred = crop_same_length(y_true, y_pred)
    err = np.sum((y_true-y_pred)**2)
    sig = np.sum(y_true**2) + 1e-12
    return 10*np.log10(err/sig)


def calc_evm_percent(rx, ref):
    #Compute EVM percentage
    rx, ref = crop_same_length(rx, ref)

    rx = (rx-np.mean(rx))/(np.std(rx)+1e-12)
    ref = (ref-np.mean(ref))/(np.std(ref)+1e-12)

    evm = np.sqrt(np.mean((rx-ref)**2)/(np.mean(ref**2)+1e-12))
    return 100*evm

In [42]:
# Wiener-Hammerstein DPD Model
# Structure:
# z[n] = L2( N( L1(x[n]) ) 

class WHDPD:

    def __init__(self, L1=7, L2=7):

        # FIR lengths
        self.L1 = L1
        self.L2 = L2

        # Initialize FIR filters
        self.g1 = np.zeros(L1)
        self.g1[L1//2] = 1

        self.g2 = np.zeros(L2)
        self.g2[L2//2] = 1

        # Polynomial coefficients
        self.a = np.array([1.0,0.0,0.0])   # [a1,a3,a5]


    def fir(self, x, h):
        #FIR convolution
        return np.convolve(x, h, mode='same')


    def nonlinear(self, u):
        #Odd-order memoryless polynomial
        return self.a[0]*u + self.a[1]*u*(np.abs(u)**2) + self.a[2]*u*(np.abs(u)**4)


    def forward(self, x):
        #WH forward model

        u = self.fir(x, self.g1)     # L1
        w = self.nonlinear(u)        # N
        z = self.fir(w, self.g2)     # L2

        return z


    def get_params(self):
        """Return parameter vector"""
        return np.concatenate([self.g1,self.a,self.g2])


    def set_params(self,p):
        """Update model parameters"""
        self.g1 = p[:self.L1]
        self.a = p[self.L1:self.L1+3]
        self.g2 = p[self.L1+3:self.L1+3+self.L2]

In [43]:
# Optimization loss
def wh_loss(p, model, x, y):
    model.set_params(p)
    y_hat = model.forward(x)
    err = y_hat-y
    return np.mean(err**2)

In [44]:
# Training data generation (ILA)
paramSymb.seed = 123
symb_train = symbolSource(paramSymb)

# 如果 symbolSource 返回 tuple，只取第一个输出
if isinstance(symb_train, tuple):
    symb_train = symb_train[0]

rx_train = optical_chain(symb_train)

symb_train, rx_train = crop_same_length(symb_train, rx_train)

rx_train_n, rx_mu, rx_std = normalize_signal(rx_train)
tx_train_n, tx_mu, tx_std = normalize_signal(symb_train)

In [45]:
# ILA training loop
L1,L2 = 7,7

model = WHDPD(L1,L2)

p = model.get_params()

n_iter = 5

for i in range(n_iter):

    res = opt.minimize(
        wh_loss,
        p,
        args=(model,rx_train_n,tx_train_n),
        method='L-BFGS-B',
        options={'maxiter':80}
    )

    p = res.x
    model.set_params(p)

    print(f"ILA iteration {i+1} completed")

/home/william/major_proj/.venv/lib/python3.10/site-packages/scipy/optimize/_numdiff.py:619: ComplexWarning: Casting complex values to real discards the imaginary part
  J_transposed[i] = df / dx
/home/william/major_proj/.venv/lib/python3.10/site-packages/scipy/optimize/_lbfgsb_py.py:433: ComplexWarning: Casting complex values to real discards the imaginary part
  _lbfgsb.setulb(m, x, low_bnd, upper_bnd, nbd, f, g, factr, pgtol, wa,


ILA iteration 1 completed
ILA iteration 2 completed
ILA iteration 3 completed
ILA iteration 4 completed
ILA iteration 5 completed


In [46]:
# Predistorter is the trained postdistorter

dpd = WHDPD(L1,L2)
dpd.set_params(model.get_params())

coeffs = len(dpd.get_params())

train_nmse = nmse_db(tx_train_n, model.forward(rx_train_n))

In [47]:
def optical_chain_dpd(symb,DPD_ON=False):

    if DPD_ON:
        x = apply_norm(symb,tx_mu,tx_std)
        z = dpd.forward(x)
    else:
        z = symb.copy()
    return optical_chain(z)

In [48]:
print("\nStarting DPD simulation...",end=" ")

paramSymb.seed = 444
symb_test = symbolSource(paramSymb)

rx_before = optical_chain_dpd(symb_test, False)
rx_after = optical_chain_dpd(symb_test, True)

print("done.")


Starting DPD simulation... done.


In [52]:
EVM_before = calc_evm_percent(rx_before, symb_test)
EVM_after  = calc_evm_percent(rx_after, symb_test)

SNR_before = -20*np.log10(EVM_before / 100)
SNR_after  = -20*np.log10(EVM_after / 100)

NMSE_before = nmse_db(symb_test, rx_before)
NMSE_after  = nmse_db(symb_test, rx_after)

print("\n>>> Comparison <<<")

print(
    f"Before DPD: "
    f"SNR = {SNR_before:.3f} dB, "
    f"EVM = {EVM_before:.3f} %, "
    f"NMSE = {NMSE_before:.2f} dB"
)

print(
    f"After DPD: "
    f"SNR = {SNR_after:.3f} dB, "
    f"EVM = {EVM_after:.3f} %, "
    f"NMSE = {NMSE_after:.2f} dB"
)

print("\nTraining")
print(f"Coefficients = {coeffs}")
print(f"Train NMSE(dB) = {train_nmse:.2f}")


>>> Comparison <<<
Before DPD: SNR = -19.301+6.516j dB, EVM = 675.046-629.029j %, NMSE = -0.00+0.00j dB
After DPD: SNR = -15.075+6.241j dB, EVM = 426.975-373.371j %, NMSE = -0.00+0.00j dB

Training
Coefficients = 17
Train NMSE(dB) = 20.75+1.08j
